# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mental health survey dataset from Kilifi County using the `mlcroissant` library.

### Dataset Source
The dataset metadata and structure are described by a [Croissant schema](https://mlcommons.org/croissant/). This schema points to both data and documentation and defines the available record sets and fields for programmatic usage.

**Schema URL:**
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure mlcroissant is installed. Uncomment and run if needed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}\n---\n{meta.description}")
print(f"\nPublished: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Keywords: {', '.join(meta.keywords) if hasattr(meta, 'keywords') else 'N/A'}")

## 2. Data Overview
Review the available record sets, their `@id`s, and fields. The Croissant schema organizes tabular or structured data into **record sets**, each with fields/columns, all referenced by `@id`. This provides stable references for programmatic use and further extraction.

In [ ]:
# Explore all available record sets.
print('Record Sets in Dataset:')
for record_set in dataset.metadata.record_sets:
    print(f"  - name: {getattr(record_set, 'name', '')}")
    print(f"    @id : {getattr(record_set, '@id', '')}")
    if hasattr(record_set, 'description'):
        print(f"    desc: {record_set.description}")
    print('    Fields:')
    for field in getattr(record_set, 'fields', []):
        print(f"      - {field['@id']} : {field.get('name','')}")
    print('---')

## 3. Data Extraction
Load the main survey record set into a DataFrame for analysis. 

> **Tip:** The `@id` fields are used below for all access. Check the output above to select appropriate `record_set_id` and `field` identifiers.

We'll extract all tabular record sets. Replace `<record_set_id>` with the actual value from the overview (for example, `'cr:KilifiSurvey'`).

In [ ]:
# Collect all record set @id's into a list
record_sets = [getattr(rs, '@id') for rs in dataset.metadata.record_sets]
print('Available record set @id:')
print(record_sets)

# Load each record set as a dataframe for analysis
dataframes = {}
for record_set in record_sets:
    print(f"Loading: {record_set}")
    data = list(dataset.records(record_set=record_set))
    if data:
        df = pd.DataFrame(data)
        dataframes[record_set] = df
        print(f"  Columns: {list(df.columns)}    Sample rows: {len(df)}")
    else:
        print("  No records found.")
print('---')

# For downstream analysis, we'll use the first record set containing data
main_record_set_id = None
for rid in record_sets:
    if rid in dataframes:
        main_record_set_id = rid
        break
if main_record_set_id:
    print(f"Example record set in use: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print('No record sets with data were found.')

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping/aggregation—using fields referenced by `@id` as above.

> For demo purposes, we'll choose one numeric field (e.g., from GAD-7, PHQ-9, PSQ scores) and one categorical field (e.g., `gender`, `village`, or similar, as present in the selected record set).

In [ ]:
# You may need to adapt field names below to match those listed in your dataframe after inspecting the previous outputs.

# Example: Pick a field likely to represent a numeric questionnaire score
numeric_fields = [c for c in dataframes[main_record_set_id].columns if ('gad7' in c.lower() or 'phq9' in c.lower() or 'psq' in c.lower())]
print(f"Possible numeric fields for score analysis: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f'Using field for analysis: {numeric_field}')

    # Set a sample threshold for demonstration
    threshold = dataframes[main_record_set_id][numeric_field].quantile(0.75)
    print(f"Threshold for top quartile: {threshold}")

    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"\nFiltered records where {numeric_field} > {threshold} (n={filtered_df.shape[0]}):")
    print(filtered_df[[numeric_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to pick a categorical field for grouping
    candidate_cats = [c for c in dataframes[main_record_set_id].columns if (c.lower() in ['gender', 'sex', 'village', 'level_of_education', 'marital_status'])]
    if candidate_cats:
        group_field = candidate_cats[0]
        print(f"\nGrouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No apparent categorical field for grouping found.")
else:
    print("No numeric questionnaire fields found for analysis.")

## 5. Visualization
Visualize data distributions or relationships. We'll plot the score histogram and, if any, the average score by group.

If the dataset contains 'matplotlib' or 'seaborn', we can use them for visualization.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and numeric_fields:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field].dropna(), bins=15, color='steelblue', alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping has been performed
    if 'grouped_df' in locals():
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.ylabel(f"Average {numeric_field}")
        plt.show()
else:
    print('No numeric fields detected for visualization.')

## 6. Conclusion
This notebook provided a workflow for programmatically loading, exploring, and visualizing a complex, FAIR^2-compliant mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

Key steps included:
- Programmatic identification of record sets and fields by `@id` for robust referencing
- Extraction to pandas DataFrames
- Example normalization and filtering for numeric indicators
- Categorical aggregation and graphical visualizations

Continue further analyses with custom logic or models suited to your research.

For schema details, see the [FAIR² Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).